Esta célula realiza a análise gráfica dos resíduos das equações preditivas de VOP, salvando automaticamente os gráficos em arquivos PNG com mesma escala entre as equações e incluindo linhas de bias e limites de concordância (±1,96 DP).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

caminho_arquivo = 'Resultados/indios_resultado.xlsx'
aba_selecionada = 'Com_Fator'
equacoes = ['Eq_ufes_sem_sexo_(+)', 'Eq_ufes_com_sexo_(+)', 'Eq_europa_(+)']
col_real = 'VOP'

pasta_saida = 'resultados_graficos'
if not os.path.exists(pasta_saida):
    os.makedirs(pasta_saida)

try:
    df = pd.read_excel(caminho_arquivo, sheet_name=aba_selecionada)

    res_cols = []
    for eq in equacoes:
        col_name = f'res_{eq}'
        df[col_name] = df[col_real] - df[eq]
        res_cols.append(col_name)

    x_min, x_max = df[col_real].min() * 0.9, df[col_real].max() * 1.1
    y_min = df[res_cols].min().min() - 2
    y_max = df[res_cols].max().max() + 2

    for col_pred in equacoes:
        residuos = df[f'res_{col_pred}']

        bias = residuos.mean()
        sd = residuos.std()
        limite_superior = bias + (1.96 * sd)
        limite_inferior = bias - (1.96 * sd)

        plt.figure(figsize=(10, 6))

        sns.scatterplot(x=df[col_real], y=residuos, s=80, alpha=0.6, edgecolors='w')

        plt.axhline(y=bias, color='red', linestyle='--', linewidth=2, label=f'BIAS: {bias:.2f}')
        plt.axhline(y=limite_superior, color='gray', linestyle=':', linewidth=1.5, label=f'IC sup (+1.96SD): {limite_superior:.2f}')
        plt.axhline(y=limite_inferior, color='gray', linestyle=':', linewidth=1.5, label=f'IC inf (-1.96SD): {limite_inferior:.2f}')
        plt.axhline(y=0, color='black', linewidth=0.8, alpha=0.3)

        plt.title(f'ANÁLISE DE RESÍDUOS: {col_pred.upper()}', fontsize=14, fontweight='bold')
        plt.xlabel('Valor Real (VOP_MEDIDO)', fontsize=12)
        plt.ylabel('Resíduo (Real - Predito)', fontsize=12)

        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)

        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()

        nome_imagem = f"residuo_{col_pred.replace('(+)', '')}.png"
        caminho_salvamento = os.path.join(pasta_saida, nome_imagem)
        plt.savefig(caminho_salvamento, dpi=300)
        print(f"Gráfico salvo em: {caminho_salvamento}")

        plt.close()

except Exception as e:
    print(f"Erro detectado: {e}")